In [12]:
import pandas as pd
import os

#### DATASET METHODS

In [2]:
def create_denormalized_tables(df_main, df_merge):
    df_main = df_main.merge(right=df_merge.reset_index(), how='left', on='SK_ID_CURR')
    return df_main


def load_all_data():
    app_train = pd.read_csv("./case/home-credit-default-risk/application_train.csv")
    app_test = pd.read_csv("./case/home-credit-default-risk/application_test.csv")
    subm = pd.read_csv("./case/home-credit-default-risk/sample_submission.csv")
    pos_cash = pd.read_csv('./case/home-credit-default-risk/POS_CASH_balance.csv')
    credit_card = pd.read_csv('./case/home-credit-default-risk/credit_card_balance.csv')
    bureau = pd.read_csv('./case/home-credit-default-risk/bureau.csv')
    bureau_balance = pd.read_csv('./case/home-credit-default-risk/bureau_balance.csv')
    previous_app = pd.read_csv('./case/home-credit-default-risk/previous_application.csv')
    install_payments = pd.read_csv('./case/home-credit-default-risk/installments_payments.csv')

    return app_train, app_test, subm, pos_cash, credit_card, bureau, bureau_balance, previous_app, install_payments

#### Step #1: Load data

In [5]:
# Load all tables
app_train, app_test, subm, pos_cash, credit_card, bureau, bureau_balance, previous_app, install_payments = load_all_data()

In [6]:
# Create denormalized tables

customer_prev_apps_count = previous_app[['SK_ID_CURR', 'SK_ID_PREV']].groupby('SK_ID_CURR').count()
previous_app['SK_ID_PREV'] = previous_app['SK_ID_CURR'].map(customer_prev_apps_count['SK_ID_PREV'])
prev_apps_avg = previous_app.groupby('SK_ID_CURR')[previous_app.select_dtypes(include='number').columns].mean()
prev_apps_avg.columns = ['P_' + col for col in prev_apps_avg.columns]
app_train = create_denormalized_tables(app_train, prev_apps_avg)
app_test = create_denormalized_tables(app_test, prev_apps_avg)

bureau_avg = bureau.groupby('SK_ID_CURR')[bureau.select_dtypes(include='number').columns].mean()
bureau_avg['BUREAU_COUNT'] = bureau[['SK_ID_BUREAU', 'SK_ID_CURR']].groupby('SK_ID_CURR').count()['SK_ID_BUREAU']
bureau_avg.columns = ['B_' + col for col in bureau_avg.columns]
app_train = create_denormalized_tables(app_train, bureau_avg)
app_test = create_denormalized_tables(app_test, bureau_avg)

install_count = install_payments[['SK_ID_CURR', 'SK_ID_PREV']].groupby('SK_ID_CURR').count()
install_payments['SK_ID_PREV'] = install_payments['SK_ID_CURR'].map(install_count['SK_ID_PREV'])
install_avg = install_payments.groupby('SK_ID_CURR')[install_payments.select_dtypes(include='number').columns].mean()
install_avg.columns = ['I_' + col for col in install_avg.columns]
app_train = create_denormalized_tables(app_train, install_avg)
app_test = create_denormalized_tables(app_test, install_avg)

prev_credit_count = credit_card[['SK_ID_CURR', 'SK_ID_PREV']].groupby('SK_ID_CURR').count()
credit_card['SK_ID_PREV'] = credit_card['SK_ID_CURR'].map(prev_credit_count['SK_ID_PREV'])
avg_credit_bal = credit_card.groupby('SK_ID_CURR')[credit_card.select_dtypes(include='number').columns].mean()
avg_credit_bal.columns = ['CC_B_' + col for col in avg_credit_bal.columns]
app_train = create_denormalized_tables(app_train, avg_credit_bal)
app_test = create_denormalized_tables(app_test, avg_credit_bal)

app_train['CREDIT_INCOME_PERCENT'] = app_train['AMT_CREDIT'] / app_train['AMT_INCOME_TOTAL']
app_train['ANNUITY_INCOME_PERCENT'] = app_train['AMT_ANNUITY'] / app_train['AMT_INCOME_TOTAL']
app_train['CREDIT_TERM'] = app_train['AMT_ANNUITY'] / app_train['AMT_CREDIT']
app_train['DAYS_EMPLOYED_PERCENT'] = app_train['DAYS_EMPLOYED'] / app_train['DAYS_BIRTH']

app_test['CREDIT_INCOME_PERCENT'] = app_test['AMT_CREDIT'] / app_test['AMT_INCOME_TOTAL']
app_test['ANNUITY_INCOME_PERCENT'] = app_test['AMT_ANNUITY'] / app_test['AMT_INCOME_TOTAL']
app_test['CREDIT_TERM'] = app_test['AMT_ANNUITY'] / app_test['AMT_CREDIT']
app_test['DAYS_EMPLOYED_PERCENT'] = app_test['DAYS_EMPLOYED'] / app_test['DAYS_BIRTH']

In [7]:
#### Step #2: Save new data

In [10]:
os.chdir("/Users/mustafaaktas/PycharmProjects/Federated-Learning-Comparative-Study/data")

In [11]:
app_train.to_csv("home_credit_default_risk_train.csv", index=False)
app_test.to_csv("home_credit_default_risk_test.csv", index=False)